In [3]:
from pathlib import Path
import sys
import time
import heapq
from collections import deque

In [4]:
AIPY_DIR = Path(""
                "aipython/aipython")

if str(AIPY_DIR) not in sys.path:
    sys.path.insert(0, str(AIPY_DIR))

from stripsProblem import Strips, STRIPS_domain, Planning_problem
from stripsForwardPlanner import Forward_STRIPS
from searchMPP import SearcherMPP

In [5]:
BLOCKS = ["a", "b", "c", "d"]
TABLES = ["t1", "t2", "t3"]
SUPPORTS = BLOCKS + TABLES

def on_feat(x):
    return f"on_{x}"

def clear_feat(x):
    return f"clear_{x}"

def empty_feat(t):
    return f"empty_{t}"

In [6]:
def complete_state(assignments, blocks=BLOCKS, tables=TABLES):
    s = {}
    for b in blocks:
        s[on_feat(b)] = None
        s[clear_feat(b)] = False
    for t in tables:
        s[empty_feat(t)] = True
    s.update(assignments)
    return s

In [7]:
def make_move_action(x, y, z):
    name = f"move({x},{y},{z})"

    pre = {
        on_feat(x): y,
        clear_feat(x): True,
    }

    if z in BLOCKS:
        pre[clear_feat(z)] = True
    else:
        pre[empty_feat(z)] = True

    eff = {
        on_feat(x): z,
    }

    if y in BLOCKS:
        eff[clear_feat(y)] = True
    else:
        eff[empty_feat(y)] = True

    if z in BLOCKS:
        eff[clear_feat(z)] = False
    else:
        eff[empty_feat(z)] = False

    return Strips(name, pre, eff, cost=1)

In [8]:
def make_feature_domains(blocks=BLOCKS, tables=TABLES):
    feats_vals = {}

    for b in blocks:
        feats_vals[on_feat(b)] = set(blocks + tables) - {b}
        feats_vals[clear_feat(b)] = {True, False}

    for t in tables:
        feats_vals[empty_feat(t)] = {True, False}

    return feats_vals

In [9]:
def make_blocks_domain(blocks=BLOCKS, tables=TABLES):
    actions = []
    supports = blocks + tables

    for x in blocks:
        for y in supports:
            if y == x:
                continue
            for z in supports:
                if z == x or z == y:
                    continue
                action = make_move_action(x, y, z)
                actions.append(action)

    feats_vals = make_feature_domains(blocks, tables)
    return STRIPS_domain(feats_vals, actions)

domain = make_blocks_domain()
len(domain.actions)

120

In [10]:
def state_to_frozenset(state_dict):
    return frozenset(state_dict.items())

def frozenset_to_state(fs):
    return dict(fs)

In [11]:
def valid_state(state, blocks=BLOCKS, tables=TABLES):
    # 1) każdy klocek ma dokładnie jedno podparcie
    supports_count = {b: 0 for b in blocks}
    for b in blocks:
        if state[on_feat(b)] in SUPPORTS:
            supports_count[b] += 1
    if any(v != 1 for v in supports_count.values()):
        return False

    # 2) brak cykli w relacji on
    for b in blocks:
        seen = set()
        cur = b
        while True:
            sup = state[on_feat(cur)]
            if sup in TABLES:
                break
            if sup in seen:
                return False
            seen.add(sup)
            cur = sup

    # 3) clear(x) zgodne z tym, czy coś leży na x
    for x in blocks:
        someone_on_x = any(state[on_feat(b)] == x for b in blocks if b != x)
        if state[clear_feat(x)] == someone_on_x:
            return False

    # 4) empty(t) zgodne z tym, czy coś stoi bezpośrednio na stole t
    for t in tables:
        someone_on_t = any(state[on_feat(b)] == t for b in blocks)
        if state[empty_feat(t)] == someone_on_t:
            return False

    return True

In [12]:
def apply_action_to_state(state, action):
    new_state = dict(state)
    for f, v in action.effects.items():
        new_state[f] = v
    return new_state

In [13]:
# Problem 1: przeniesienie wieży a-b-c z t1 na t2, d stoi osobno
problem1_init = complete_state({
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t1",
    on_feat("d"): "t3",
    clear_feat("a"): True,
    clear_feat("b"): False,
    clear_feat("c"): False,
    clear_feat("d"): True,
    empty_feat("t1"): False,
    empty_feat("t2"): True,
    empty_feat("t3"): False,
})

problem1_goal = {
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t2",
}

problem1 = Planning_problem(domain, problem1_init, problem1_goal)

In [14]:
# Problem 2: odwrócenie wieży a-b-c na c-b-a i przeniesienie na t2
problem2_init = complete_state({
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t1",
    on_feat("d"): "t3",
    clear_feat("a"): True,
    clear_feat("b"): False,
    clear_feat("c"): False,
    clear_feat("d"): True,
    empty_feat("t1"): False,
    empty_feat("t2"): True,
    empty_feat("t3"): False,
})

problem2_goal = {
    on_feat("a"): "d",
    on_feat("d"): "c",
    on_feat("c"): "b",
    on_feat("b"): "t2",
}

problem2 = Planning_problem(domain, problem2_init, problem2_goal)

In [15]:
# Problem 3: wariant Sussman-like
problem3_init = complete_state({
    on_feat("a"): "t1",
    on_feat("b"): "t2",
    on_feat("c"): "a",
    on_feat("d"): "t3",
    clear_feat("a"): False,
    clear_feat("b"): True,
    clear_feat("c"): True,
    clear_feat("d"): True,
    empty_feat("t1"): False,
    empty_feat("t2"): False,
    empty_feat("t3"): False,
})

problem3_goal = {
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "t2",
}

problem3 = Planning_problem(domain, problem3_init, problem3_goal)

In [16]:
problems = {
    "problem1": problem1,
    "problem2": problem2,
    "problem3": problem3,
}

for name, p in problems.items():
    print(name, valid_state(p.initial_state))

problem1 True
problem2 True
problem3 True


In [17]:
def get_problem_domain(problem):
    if hasattr(problem, "prob_domain"):
        return problem.prob_domain
    if hasattr(problem, "domain"):
        return problem.domain
    raise AttributeError(f"Nie znalazłem domeny w Planning_problem. Atrybuty: {dir(problem)}")

In [18]:
def count_reachable_states(problem, timeout_seconds=5):
    start = time.perf_counter()

    domain = get_problem_domain(problem)
    init_fs = state_to_frozenset(problem.initial_state)

    visited = {init_fs}
    q = deque([problem.initial_state])

    while q and time.perf_counter() - start < timeout_seconds:
        state = q.popleft()

        for action in domain.actions:
            ok = True
            for f, v in action.preconds.items():
                if state[f] != v:
                    ok = False
                    break
            if not ok:
                continue

            ns = apply_action_to_state(state, action)
            if not valid_state(ns):
                continue

            fs = state_to_frozenset(ns)
            if fs not in visited:
                visited.add(fs)
                q.append(ns)

    return len(visited)

for name, p in problems.items():
    print(name, count_reachable_states(p, timeout_seconds=5))

problem1 360
problem2 360
problem3 360


In [19]:
# Forward planning w AIPython
def solve_with_forward_planner(problem):
    planner = Forward_STRIPS(problem)
    searcher = SearcherMPP(planner)
    path = searcher.search()
    return planner, searcher, path

In [20]:
def extract_actions_from_path(path):
    if path is None:
        return None

    actions = []
    cur = path
    while hasattr(cur, "arc") and cur.arc is not None:
        actions.append(cur.arc.action)
        cur = cur.initial
    actions.reverse()
    return actions

In [21]:
forward_results = {}

for name, p in problems.items():
    start = time.perf_counter()
    planner, searcher, path = solve_with_forward_planner(p)
    elapsed = time.perf_counter() - start
    plan = extract_actions_from_path(path)

    forward_results[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": getattr(searcher, "num_expanded", None),
    }

    print("=" * 80)
    print(name)
    print("time_s:", round(elapsed, 4))
    print("expanded:", getattr(searcher, "num_expanded", None))
    print("plan_len:", None if plan is None else len(plan))
    print("plan:", plan)

Solution: {'on_a': 'b', 'clear_a': True, 'on_b': 'c', 'clear_b': False, 'on_c': 't1', 'clear_c': False, 'on_d': 't3', 'clear_d': True, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(a,b,d)--> {'on_a': 'd', 'clear_a': True, 'on_b': 'c', 'clear_b': True, 'on_c': 't1', 'clear_c': False, 'on_d': 't3', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(b,c,a)--> {'on_a': 'd', 'clear_a': False, 'on_b': 'a', 'clear_b': True, 'on_c': 't1', 'clear_c': True, 'on_d': 't3', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(c,t1,t2)--> {'on_a': 'd', 'clear_a': False, 'on_b': 'a', 'clear_b': True, 'on_c': 't2', 'clear_c': True, 'on_d': 't3', 'clear_d': False, 'empty_t1': True, 'empty_t2': False, 'empty_t3': False}
   --move(b,a,c)--> {'on_a': 'd', 'clear_a': True, 'on_b': 'c', 'clear_b': True, 'on_c': 't2', 'clear_c': False, 'on_d': 't3', 'clear_d': False, 'empty_t1': True, 'empty_t2': False, 'empty_t3': False}
   

In [22]:
# Heurystyka:
# liczba niespełnionych relacji celu + kara za "zły support" dla klocków
def blocks_heuristic(state, goal):
    h = 0
    for f, v in goal.items():
        if state[f] != v:
            h += 1

    # dodatkowa lekka kara za klocki, które stoją na złym supportcie
    for b in BLOCKS:
        feat = on_feat(b)
        if feat in goal and state[feat] != goal[feat]:
            h += 1

    return h

In [23]:
def astar_problem(problem, heuristic_fn, timeout_seconds=300):
    start_time = time.perf_counter()

    domain = get_problem_domain(problem)
    start_state = problem.initial_state
    start_fs = state_to_frozenset(start_state)

    open_heap = []
    counter = 0

    g_score = {start_fs: 0}
    start_h = heuristic_fn(start_state, problem.goal)
    heapq.heappush(open_heap, (start_h, counter, start_state, []))

    closed = set()
    expanded = 0

    while open_heap:
        if time.perf_counter() - start_time > timeout_seconds:
            return None, time.perf_counter() - start_time, expanded

        f, _, state, plan = heapq.heappop(open_heap)
        state_fs = state_to_frozenset(state)

        if state_fs in closed:
            continue
        closed.add(state_fs)
        expanded += 1

        goal_ok = True
        for feat, val in problem.goal.items():
            if state[feat] != val:
                goal_ok = False
                break
        if goal_ok:
            return plan, time.perf_counter() - start_time, expanded

        for action in domain.actions:
            ok = True
            for feat, val in action.preconds.items():
                if state[feat] != val:
                    ok = False
                    break
            if not ok:
                continue

            ns = apply_action_to_state(state, action)
            if not valid_state(ns):
                continue

            ns_fs = state_to_frozenset(ns)
            tentative_g = g_score[state_fs] + 1

            if ns_fs not in g_score or tentative_g < g_score[ns_fs]:
                g_score[ns_fs] = tentative_g
                counter += 1
                h = heuristic_fn(ns, problem.goal)
                heapq.heappush(open_heap, (tentative_g + h, counter, ns, plan + [action.name]))

    return None, time.perf_counter() - start_time, expanded

In [24]:
heur_results = {}

for name, p in problems.items():
    plan, elapsed, expanded = astar_problem(p, blocks_heuristic, timeout_seconds=300)

    heur_results[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": expanded,
    }

    print("=" * 80)
    print(name)
    print("time_s:", round(elapsed, 4))
    print("expanded:", expanded)
    print("plan_len:", None if plan is None else len(plan))
    print("plan:", plan)

problem1
time_s: 0.0031
expanded: 16
plan_len: 5
plan: ['move(a,b,d)', 'move(b,c,a)', 'move(c,t1,t2)', 'move(b,a,c)', 'move(a,d,b)']
problem2
time_s: 0.0017
expanded: 11
plan_len: 6
plan: ['move(a,b,d)', 'move(b,c,t2)', 'move(c,t1,b)', 'move(a,d,t1)', 'move(d,t3,c)', 'move(a,t1,d)']
problem3
time_s: 0.0019
expanded: 9
plan_len: 4
plan: ['move(b,t2,d)', 'move(c,a,t2)', 'move(b,d,c)', 'move(a,t1,b)']


In [25]:
heur_results = {}

for name, p in problems.items():
    plan, elapsed, expanded = astar_problem(p, blocks_heuristic, timeout_seconds=300)

    heur_results[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": expanded,
    }

    print("=" * 80)
    print(name)
    print("time_s:", round(elapsed, 4))
    print("expanded:", expanded)
    print("plan_len:", None if plan is None else len(plan))
    print("plan:", plan)

problem1
time_s: 0.0058
expanded: 16
plan_len: 5
plan: ['move(a,b,d)', 'move(b,c,a)', 'move(c,t1,t2)', 'move(b,a,c)', 'move(a,d,b)']
problem2
time_s: 0.0022
expanded: 11
plan_len: 6
plan: ['move(a,b,d)', 'move(b,c,t2)', 'move(c,t1,b)', 'move(a,d,t1)', 'move(d,t3,c)', 'move(a,t1,d)']
problem3
time_s: 0.0015
expanded: 9
plan_len: 4
plan: ['move(b,t2,d)', 'move(c,a,t2)', 'move(b,d,c)', 'move(a,t1,b)']


In [26]:
import pandas as pd

rows = []
for name in problems:
    rows.append({
        "problem": name,
        "reachable_states_5s": count_reachable_states(problems[name], 5),
        "forward_plan_len": None if forward_results[name]["plan"] is None else len(forward_results[name]["plan"]),
        "forward_time_s": round(forward_results[name]["time_s"], 4),
        "forward_expanded": forward_results[name]["expanded"],
        "heur_plan_len": None if heur_results[name]["plan"] is None else len(heur_results[name]["plan"]),
        "heur_time_s": round(heur_results[name]["time_s"], 4),
        "heur_expanded": heur_results[name]["expanded"],
        "heur_plan": heur_results[name]["plan"],
    })

df = pd.DataFrame(rows)
df

,problem,reachable_states_5s,forward_plan_len,forward_time_s,forward_expanded,heur_plan_len,heur_time_s,heur_expanded,heur_plan
0,problem1,360,5,0.0639,144,5,0.0058,16,"[move(a,b,d), move(b,c,a), move(c,t1,t2), move..."
1,problem2,360,6,0.1090,177,6,0.0022,11,"[move(a,b,d), move(b,c,t2), move(c,t1,b), move..."
2,problem3,360,4,0.0241,62,4,0.0015,9,"[move(b,t2,d), move(c,a,t2), move(b,d,c), move..."


In [27]:
# Ładny wydruk planów do sprawozdania
for name in problems:
    print("=" * 100)
    print(name)
    print("Forward planning:")
    if forward_results[name]["plan"] is not None:
        for i, a in enumerate(forward_results[name]["plan"], 1):
            print(f"{i}. {a}")
    else:
        print("brak rozwiązania")

    print("\nZ heurystyką:")
    if heur_results[name]["plan"] is not None:
        for i, a in enumerate(heur_results[name]["plan"], 1):
            print(f"{i}. {a}")
    else:
        print("brak rozwiązania")

    print("\nCzas z heurystyką:", round(heur_results[name]["time_s"], 4), "s")

problem1
Forward planning:
1. move(a,b,d)
2. move(b,c,a)
3. move(c,t1,t2)
4. move(b,a,c)
5. move(a,d,b)

Z heurystyką:
1. move(a,b,d)
2. move(b,c,a)
3. move(c,t1,t2)
4. move(b,a,c)
5. move(a,d,b)

Czas z heurystyką: 0.0058 s
problem2
Forward planning:
1. move(a,b,d)
2. move(b,c,t2)
3. move(c,t1,b)
4. move(a,d,t1)
5. move(d,t3,c)
6. move(a,t1,d)

Z heurystyką:
1. move(a,b,d)
2. move(b,c,t2)
3. move(c,t1,b)
4. move(a,d,t1)
5. move(d,t3,c)
6. move(a,t1,d)

Czas z heurystyką: 0.0022 s
problem3
Forward planning:
1. move(b,t2,d)
2. move(c,a,t2)
3. move(b,d,c)
4. move(a,t1,b)

Z heurystyką:
1. move(b,t2,d)
2. move(c,a,t2)
3. move(b,d,c)
4. move(a,t1,b)

Czas z heurystyką: 0.0015 s


In [28]:
# Krótka kontrola warunku z treści zadania:
for name in problems:
    rs = count_reachable_states(problems[name], 5)
    fp = forward_results[name]["plan"]
    hp = heur_results[name]["plan"]
    print(name, {
        "reachable_states_5s": rs,
        "forward_len": None if fp is None else len(fp),
        "heur_len": None if hp is None else len(hp),
    })

problem1 {'reachable_states_5s': 360, 'forward_len': 5, 'heur_len': 5}
problem2 {'reachable_states_5s': 360, 'forward_len': 6, 'heur_len': 6}
problem3 {'reachable_states_5s': 360, 'forward_len': 4, 'heur_len': 4}


## Zadanie za 6 punktow

In [29]:
import io
import time
import contextlib
import pandas as pd

subgoals = {
    "problem1": [
        {on_feat("a"): "d", on_feat("b"): "a"},
        {on_feat("a"): "d", on_feat("b"): "a", on_feat("c"): "t2"},
    ],
    "problem2": [
        {on_feat("b"): "t2"},
        {on_feat("b"): "t2", on_feat("c"): "b"},
    ],
    "problem3": [
        {on_feat("b"): "d"},
        {on_feat("b"): "d", on_feat("c"): "t2"},
    ],
}

def normalize_action_name(action_or_name):
    return action_or_name if isinstance(action_or_name, str) else action_or_name.name

def normalize_plan(plan):
    if plan is None:
        return None
    return [normalize_action_name(step) for step in plan]

def apply_plan_from_state(state, plan, domain):
    actions_by_name = {action.name: action for action in domain.actions}
    current_state = dict(state)

    for step in plan or []:
        current_state = apply_action_to_state(
            current_state, actions_by_name[normalize_action_name(step)]
        )

    return current_state

def solve_forward_quiet(problem):
    start = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        planner, searcher, path = solve_with_forward_planner(problem)
    elapsed = time.perf_counter() - start

    plan = normalize_plan(extract_actions_from_path(path))
    end_state = (
        dict(problem.initial_state)
        if plan is None
        else apply_plan_from_state(problem.initial_state, plan, get_problem_domain(problem))
    )

    return {
        "plan": plan,
        "time_s": elapsed,
        "expanded": getattr(searcher, "num_expanded", None),
        "end_state": end_state,
    }

def solve_astar_quiet(problem):
    plan, elapsed, expanded = astar_problem(problem, blocks_heuristic, timeout_seconds=300)
    end_state = (
        dict(problem.initial_state)
        if plan is None
        else apply_plan_from_state(problem.initial_state, plan, get_problem_domain(problem))
    )

    return {
        "plan": plan,
        "time_s": elapsed,
        "expanded": expanded,
        "end_state": end_state,
    }

def solve_with_subgoals(problem, stage_goals, solver_fn):
    current_state = dict(problem.initial_state)
    full_plan = []
    total_time = 0.0
    total_expanded = 0
    stages = []

    for stage_idx, goal in enumerate(stage_goals + [problem.goal], start=1):
        subproblem = Planning_problem(get_problem_domain(problem), current_state, goal)
        result = solver_fn(subproblem)

        stages.append({
            "stage": stage_idx,
            "goal": goal,
            "plan": result["plan"],
            "time_s": result["time_s"],
            "expanded": result["expanded"],
        })

        total_time += result["time_s"]
        total_expanded += result["expanded"] or 0

        if result["plan"] is None:
            return {
                "plan": None,
                "time_s": total_time,
                "expanded": total_expanded,
                "stages": stages,
            }

        full_plan.extend(result["plan"])
        current_state = result["end_state"]

    return {
        "plan": full_plan,
        "time_s": total_time,
        "expanded": total_expanded,
        "stages": stages,
    }

def goal_to_text(goal):
    return ", ".join(f"{feat}={value}" for feat, value in sorted(goal.items()))

subgoal_results = {}

for name, problem in problems.items():
    subgoal_results[name] = {
        "forward": solve_with_subgoals(problem, subgoals[name], solve_forward_quiet),
        "heur": solve_with_subgoals(problem, subgoals[name], solve_astar_quiet),
    }

for name in problems:
    print("=" * 100)
    print(name)
    print("Podcele:")
    for idx, goal in enumerate(subgoals[name], start=1):
        print(f"{idx}. {goal_to_text(goal)}")

    for label, method in [("Forward + podcele", "forward"), ("Heurystyka + podcele", "heur")]:
        result = subgoal_results[name][method]
        print(f"\n{label}:")
        for stage in result["stages"]:
            stage_name = (
                "cel końcowy"
                if stage["stage"] == len(subgoals[name]) + 1
                else f"podcel {stage['stage']}"
            )
            print(f"{stage_name}: {goal_to_text(stage['goal'])}")
            print("plan:", "brak" if stage["plan"] is None else ", ".join(stage["plan"]))
            print("długość:", None if stage["plan"] is None else len(stage["plan"]))
            print("time_s:", round(stage["time_s"], 4), "expanded:", stage["expanded"])

        print("plan łączny:", "brak" if result["plan"] is None else ", ".join(result["plan"]))
        print("łączna długość:", None if result["plan"] is None else len(result["plan"]))
        print("łączny czas:", round(result["time_s"], 4), "s")
        print("łącznie expanded:", result["expanded"])

summary_rows = []
for name in problems:
    base_forward_plan = normalize_plan(forward_results[name]["plan"])
    base_heur_plan = normalize_plan(heur_results[name]["plan"])

    sub_forward = subgoal_results[name]["forward"]
    sub_heur = subgoal_results[name]["heur"]

    summary_rows.append({
        "problem": name,

        "base_forward_len": None if base_forward_plan is None else len(base_forward_plan),
        "base_forward_time_s": round(forward_results[name]["time_s"], 4),
        "base_forward_expanded": forward_results[name]["expanded"],

        "subgoal_forward_len": None if sub_forward["plan"] is None else len(sub_forward["plan"]),
        "subgoal_forward_time_s": round(sub_forward["time_s"], 4),
        "subgoal_forward_expanded": sub_forward["expanded"],

        "base_heur_len": None if base_heur_plan is None else len(base_heur_plan),
        "base_heur_time_s": round(heur_results[name]["time_s"], 4),
        "base_heur_expanded": heur_results[name]["expanded"],

        "subgoal_heur_len": None if sub_heur["plan"] is None else len(sub_heur["plan"]),
        "subgoal_heur_time_s": round(sub_heur["time_s"], 4),
        "subgoal_heur_expanded": sub_heur["expanded"],

        "same_as_base_forward": sub_forward["plan"] == base_forward_plan,
        "same_as_base_heur": sub_heur["plan"] == base_heur_plan,

        "forward_time_diff_s": round(sub_forward["time_s"] - forward_results[name]["time_s"], 4),
        "heur_time_diff_s": round(sub_heur["time_s"] - heur_results[name]["time_s"], 4),
    })

summary_compare_df = pd.DataFrame(summary_rows)
summary_compare_df



problem1
Podcele:
1. on_a=d, on_b=a
2. on_a=d, on_b=a, on_c=t2

Forward + podcele:
podcel 1: on_a=d, on_b=a
plan: move(a,b,d), move(b,c,a)
długość: 2
time_s: 0.0037 expanded: 7
podcel 2: on_a=d, on_b=a, on_c=t2
plan: move(c,t1,t2)
długość: 1
time_s: 0.0005 expanded: 2
cel końcowy: on_a=b, on_b=c, on_c=t2
plan: move(b,a,c), move(a,d,b)
długość: 2
time_s: 0.0028 expanded: 7
plan łączny: move(a,b,d), move(b,c,a), move(c,t1,t2), move(b,a,c), move(a,d,b)
łączna długość: 5
łączny czas: 0.007 s
łącznie expanded: 16

Heurystyka + podcele:
podcel 1: on_a=d, on_b=a
plan: move(a,b,d), move(b,c,a)
długość: 2
time_s: 0.001 expanded: 3
podcel 2: on_a=d, on_b=a, on_c=t2
plan: move(c,t1,t2)
długość: 1
time_s: 0.0004 expanded: 2
cel końcowy: on_a=b, on_b=c, on_c=t2
plan: move(b,a,c), move(a,d,b)
długość: 2
time_s: 0.0007 expanded: 3
plan łączny: move(a,b,d), move(b,c,a), move(c,t1,t2), move(b,a,c), move(a,d,b)
łączna długość: 5
łączny czas: 0.002 s
łącznie expanded: 8
problem2
Podcele:
1. on_b=t2
2. on

,problem,base_forward_len,base_forward_time_s,base_forward_expanded,subgoal_forward_len,subgoal_forward_time_s,subgoal_forward_expanded,base_heur_len,base_heur_time_s,base_heur_expanded,subgoal_heur_len,subgoal_heur_time_s,subgoal_heur_expanded,same_as_base_forward,same_as_base_heur,forward_time_diff_s,heur_time_diff_s
0,problem1,5,0.0639,144,5,0.0070,16,5,0.0058,16,5,0.0020,8,True,True,-0.0570,-0.0038
1,problem2,6,0.1090,177,6,0.0253,50,6,0.0022,11,6,0.0026,10,True,True,-0.0837,0.0004
2,problem3,4,0.0241,62,4,0.0118,27,4,0.0015,9,4,0.0020,8,True,True,-0.0123,0.0005


## Problemy wymagające minimum 20 akcji (z podcelami)

### Problem new1 – Hanoi i odwrócenie
- **Stan początkowy:** pełna wieża a-b-c-d na t1 (a na wierzchu, d na dole); t2 i t3 wolne
- **Podcel 1:** przeniesienie całej wieży (w tej samej kolejności) na t2 — ruch Hanoï, ok. 15 kroków
- **Cel końcowy:** odwrócona wieża na t3 (a na dole, d na wierzchu) — ok. 10 kroków
- **Szacowana długość planu z podcelami:** ~25 kroków

### Problem new2 – Podwójne odwrócenie z Hanoï
- **Stan początkowy:** pełna wieża a-b-c-d na t1 (a na wierzchu); t2 i t3 wolne
- **Podcel 1:** odwrócona wieża na t2 (a na dole, d na wierzchu) — ok. 10 kroków
- **Podcel 2:** ta sama odwrócona wieża przeniesiona na t1 — ruch Hanoï, ok. 15 kroków
- **Cel końcowy:** wieża w oryginalnej kolejności na t3 (d na dole, a na wierzchu) — ok. 10 kroków
- **Szacowana długość planu z podcelami:** ~35 kroków

### Problem new3 – Scalenie i Hanoï
- **Stan początkowy:** dwie wieże 2-klockowe: b nad a na t1 oraz d nad c na t2; t3 wolne
- **Podcel 1:** złożenie jednej wieży a-b-c-d na t1 (a na dole, d na wierzchu) — ok. 3 kroki
- **Podcel 2:** przeniesienie całej wieży na t2 — ruch Hanoï, ok. 15 kroków
- **Cel końcowy:** wieża w oryginalnej kolejności na t1 (d na dole, a na wierzchu) — ok. 10 kroków
- **Szacowana długość planu z podcelami:** ~28 kroków

In [30]:
# ── Definicja 3 nowych problemów (zad na 8) ──────────────────────────────────

# Problem new1: pełna wieża a-b-c-d na t1, podcel = Hanoï na t2, cel = odwrócona na t3
problem_new1_init = complete_state({
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "d",
    on_feat("d"): "t1",
    clear_feat("a"): True,
    clear_feat("b"): False,
    clear_feat("c"): False,
    clear_feat("d"): False,
    empty_feat("t1"): False,
    empty_feat("t2"): True,
    empty_feat("t3"): True,
})
problem_new1_goal = {
    on_feat("a"): "t3",
    on_feat("b"): "a",
    on_feat("c"): "b",
    on_feat("d"): "c",
}
problem_new1 = Planning_problem(domain, problem_new1_init, problem_new1_goal)

# Problem new2: pełna wieża a-b-c-d na t1, podcel1 = odwrócona na t2,
#               podcel2 = odwrócona przeniesiona na t1 (Hanoï), cel = oryginalna na t3
problem_new2_init = complete_state({
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "d",
    on_feat("d"): "t1",
    clear_feat("a"): True,
    clear_feat("b"): False,
    clear_feat("c"): False,
    clear_feat("d"): False,
    empty_feat("t1"): False,
    empty_feat("t2"): True,
    empty_feat("t3"): True,
})
problem_new2_goal = {
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "d",
    on_feat("d"): "t3",
}
problem_new2 = Planning_problem(domain, problem_new2_init, problem_new2_goal)

# Problem new3: dwie wieże 2-klockowe (b/a na t1, d/c na t2),
#               podcel1 = scal na t1, podcel2 = Hanoï na t2, cel = oryginalna kolejność na t1
problem_new3_init = complete_state({
    on_feat("a"): "t1",
    on_feat("b"): "a",
    on_feat("c"): "t2",
    on_feat("d"): "c",
    clear_feat("a"): False,
    clear_feat("b"): True,
    clear_feat("c"): False,
    clear_feat("d"): True,
    empty_feat("t1"): False,
    empty_feat("t2"): False,
    empty_feat("t3"): True,
})
problem_new3_goal = {
    on_feat("a"): "b",
    on_feat("b"): "c",
    on_feat("c"): "d",
    on_feat("d"): "t1",
}
problem_new3 = Planning_problem(domain, problem_new3_init, problem_new3_goal)

problems_new = {
    "problem_new1": problem_new1,
    "problem_new2": problem_new2,
    "problem_new3": problem_new3,
}

# Weryfikacja poprawności stanów początkowych
for name, p in problems_new.items():
    print(name, "valid:", valid_state(p.initial_state))

# PODCELE
subgoals_new = {
    # Podcel: ta sama wieża na t2 (ruch Hanoi ~15 kroków)
    "problem_new1": [
        {on_feat("a"): "b", on_feat("b"): "c", on_feat("c"): "d", on_feat("d"): "t2"},
    ],
    # Podcel 1: odwrócona wieża na t2 (~10 kroków)
    # Podcel 2: odwrócona wieża przeniesiona na t1 (Hanoï ~15 kroków)
    "problem_new2": [
        {on_feat("a"): "t2", on_feat("b"): "a", on_feat("c"): "b", on_feat("d"): "c"},
        {on_feat("a"): "t1", on_feat("b"): "a", on_feat("c"): "b", on_feat("d"): "c"},
    ],
    # Podcel 1: złożenie wieży na t1 (~3 kroki)
    # Podcel 2: Hanoï na t2 (~15 kroków)
    "problem_new3": [
        {on_feat("a"): "t1", on_feat("b"): "a", on_feat("c"): "b", on_feat("d"): "c"},
        {on_feat("a"): "t2", on_feat("b"): "a", on_feat("c"): "b", on_feat("d"): "c"},
    ],
}

problem_new1 valid: True
problem_new2 valid: True
problem_new3 valid: True


In [31]:
# ── Rozwiązanie nowych problemów (analogicznie do powyższych) ─────────────────

forward_results_new = {}
for name, p in problems_new.items():
    start = time.perf_counter()
    planner, searcher, path = solve_with_forward_planner(p)
    elapsed = time.perf_counter() - start
    plan = normalize_plan(extract_actions_from_path(path))
    forward_results_new[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": getattr(searcher, "num_expanded", None),
    }

heur_results_new = {}
for name, p in problems_new.items():
    plan, elapsed, expanded = astar_problem(p, blocks_heuristic, timeout_seconds=300)
    heur_results_new[name] = {
        "plan": plan,
        "time_s": elapsed,
        "expanded": expanded,
    }

subgoal_results_new = {}
for name, p in problems_new.items():
    subgoal_results_new[name] = {
        "forward": solve_with_subgoals(p, subgoals_new[name], solve_forward_quiet),
        "heur":    solve_with_subgoals(p, subgoals_new[name], solve_astar_quiet),
    }
# ------------------------------------------------------------------------------------------------
# szczegółowy output
for name in problems_new:
    print("=" * 100)
    print(name)
    print("Forward (bez podceli):")
    fp = forward_results_new[name]["plan"]
    if fp:
        for i, a in enumerate(fp, 1):
            print(f"  {i}. {a}")
    else:
        print("  brak rozwiązania")
    print(f"  długość={None if fp is None else len(fp)}, "
          f"czas={round(forward_results_new[name]['time_s'],4)} s, "
          f"expanded={forward_results_new[name]['expanded']}")

    print("Heurystyka A* (bez podceli):")
    hp = heur_results_new[name]["plan"]
    if hp:
        for i, a in enumerate(hp, 1):
            print(f"  {i}. {a}")
    else:
        print("  brak rozwiązania")
    print(f"  długość={None if hp is None else len(hp)}, "
          f"czas={round(heur_results_new[name]['time_s'],4)} s, "
          f"expanded={heur_results_new[name]['expanded']}")

    for label, method in [("Forward + podcele", "forward"), ("Heurystyka + podcele", "heur")]:
        result = subgoal_results_new[name][method]
        print(f"\n{label}:")
        for stage in result["stages"]:
            stage_name = (
                "cel końcowy"
                if stage["stage"] == len(subgoals_new[name]) + 1
                else f"podcel {stage['stage']}"
            )
            print(f"  {stage_name}: {goal_to_text(stage['goal'])}")
            print(f"    plan: {'brak' if stage['plan'] is None else ', '.join(stage['plan'])}")
            print(f"    długość={None if stage['plan'] is None else len(stage['plan'])}, "
                  f"czas={round(stage['time_s'],4)} s, expanded={stage['expanded']}")
        full = result["plan"]
        print(f"  → PLAN ŁĄCZNY (długość={None if full is None else len(full)}): "
              f"{'brak' if full is None else ', '.join(full)}")
        print(f"  łączny czas={round(result['time_s'],4)} s, łącznie expanded={result['expanded']}")


Solution: {'on_a': 'b', 'clear_a': True, 'on_b': 'c', 'clear_b': False, 'on_c': 'd', 'clear_c': False, 'on_d': 't1', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': True}
   --move(a,b,t3)--> {'on_a': 't3', 'clear_a': True, 'on_b': 'c', 'clear_b': True, 'on_c': 'd', 'clear_c': False, 'on_d': 't1', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(b,c,a)--> {'on_a': 't3', 'clear_a': False, 'on_b': 'a', 'clear_b': True, 'on_c': 'd', 'clear_c': True, 'on_d': 't1', 'clear_d': False, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(c,d,b)--> {'on_a': 't3', 'clear_a': False, 'on_b': 'a', 'clear_b': False, 'on_c': 'b', 'clear_c': True, 'on_d': 't1', 'clear_d': True, 'empty_t1': False, 'empty_t2': True, 'empty_t3': False}
   --move(d,t1,c)--> {'on_a': 't3', 'clear_a': False, 'on_b': 'a', 'clear_b': False, 'on_c': 'b', 'clear_c': False, 'on_d': 'c', 'clear_d': True, 'empty_t1': True, 'empty_t2': True, 'empty_t3': False} (cost

In [32]:
import pandas as pd

summary_rows_new = []
for name in problems_new:
    fp  = forward_results_new[name]["plan"]
    hp  = heur_results_new[name]["plan"]
    sfp = subgoal_results_new[name]["forward"]["plan"]
    shp = subgoal_results_new[name]["heur"]["plan"]
    summary_rows_new.append({
        "problem": name,
        "forward_len":         None if fp  is None else len(fp),
        "forward_time_s":      round(forward_results_new[name]["time_s"], 4),
        "forward_expanded":    forward_results_new[name]["expanded"],
        "heur_len":            None if hp  is None else len(hp),
        "heur_time_s":         round(heur_results_new[name]["time_s"], 4),
        "heur_expanded":       heur_results_new[name]["expanded"],
        "subgoal_forward_len": None if sfp is None else len(sfp),
        "subgoal_forward_time_s": round(subgoal_results_new[name]["forward"]["time_s"], 4),
        "subgoal_heur_len":    None if shp is None else len(shp),
        "subgoal_heur_time_s": round(subgoal_results_new[name]["heur"]["time_s"], 4),
    })

pd.DataFrame(summary_rows_new)

,problem,forward_len,forward_time_s,forward_expanded,heur_len,heur_time_s,heur_expanded,subgoal_forward_len,subgoal_forward_time_s,subgoal_heur_len,subgoal_heur_time_s
0,problem_new1,4,0.0273,48,4,0.0009,5,11,0.1170,11,0.0064
1,problem_new2,7,0.1763,266,7,0.0045,24,15,0.1459,15,0.0078
2,problem_new3,7,0.1741,320,7,0.0041,25,14,0.1044,14,0.0068
